# BiomedBERT MTSamples Classifier - Colab

This notebook fine-tunes `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext` on the frozen MTSamples split.

Protocol:
- Train/tune only on `data/splits/train.csv`.
- Do not use `data/splits/val.csv` for training or early stopping.
- Predict on `data/splits/val.csv` at the end.
- Export `validation_predictions.csv` with exactly `id,prediction`.

In Colab, set Runtime -> Change runtime type -> GPU before running.

In [ ]:
!pip -q install transformers accelerate scikit-learn pandas numpy

In [ ]:
from pathlib import Path
import json
import os
import random

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Load Frozen Split

Upload your repo's `data/splits/train.csv` and `data/splits/val.csv` into Colab, preserving this path:

```text
data/splits/train.csv
data/splits/val.csv
```

Fastest Colab option: use the file browser on the left to create folders and upload the two CSVs. If your project is in Google Drive, mount Drive and set `PROJECT_DIR` below.

In [ ]:
# Option A: files uploaded directly into the Colab runtime.
PROJECT_DIR = Path('.')

# Option B: uncomment if your repo/data lives in Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_DIR = Path('/content/drive/MyDrive/Codamatrix')

TRAIN_FILE = PROJECT_DIR / 'data/splits/train.csv'
VAL_FILE = PROJECT_DIR / 'data/splits/val.csv'

assert TRAIN_FILE.exists(), f'Missing {TRAIN_FILE}'
assert VAL_FILE.exists(), f'Missing {VAL_FILE}'

train_raw = pd.read_csv(TRAIN_FILE)
val_raw = pd.read_csv(VAL_FILE)

print(TRAIN_FILE, train_raw.shape, train_raw.columns.tolist())
print(VAL_FILE, val_raw.shape, val_raw.columns.tolist())
display(train_raw.head())

In [ ]:
CLASS_TO_LABEL = {
    'cardiology': 0,
    'neurology': 1,
    'orthopedics': 2,
    'gastroenterology': 3,
    'other': 4,
}
ID_TO_CLASS = {v: k for k, v in CLASS_TO_LABEL.items()}
DISPLAY_CLASS_NAMES = {
    'cardiology': 'Cardiology',
    'neurology': 'Neurology',
    'orthopedics': 'Orthopedics',
    'gastroenterology': 'Gastroenterology',
    'other': 'Other',
}
ID_TO_DISPLAY_CLASS = {i: DISPLAY_CLASS_NAMES[name] for i, name in ID_TO_CLASS.items()}
DISPLAY_CLASS_ORDER = [ID_TO_DISPLAY_CLASS[i] for i in sorted(ID_TO_DISPLAY_CLASS)]

def normalize_split(df: pd.DataFrame) -> pd.DataFrame:
    required = {'id', 'text', 'label'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'Missing columns: {sorted(missing)}')
    out = df.copy()
    out['id'] = out['id'].astype(str)
    out['encoder_text'] = out['text'].fillna('').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
    out['class_norm'] = out['label'].astype(str).str.strip().str.lower()
    out['hackathon_label'] = out['class_norm'].map(CLASS_TO_LABEL)
    bad = out[out['hackathon_label'].isna()]['label'].unique()
    if len(bad):
        raise ValueError(f'Unknown labels: {bad}')
    out = out[out['encoder_text'] != ''].copy()
    out['hackathon_label'] = out['hackathon_label'].astype(int)
    return out

frozen_train_df = normalize_split(train_raw)
final_val_df = normalize_split(val_raw)

print('Frozen train counts:')
print(frozen_train_df['class_norm'].value_counts().reindex(CLASS_TO_LABEL.keys(), fill_value=0))
print('\nFrozen val counts:')
print(final_val_df['class_norm'].value_counts().reindex(CLASS_TO_LABEL.keys(), fill_value=0))

## Internal Tune Split

`val.csv` is final scoring only. This cell creates a small internal evaluation split from `train.csv` for epoch metrics and early stopping.

In [ ]:
TUNE_SIZE = 0.15

model_train_df, internal_eval_df = train_test_split(
    frozen_train_df,
    test_size=TUNE_SIZE,
    random_state=SEED,
    stratify=frozen_train_df['hackathon_label'],
)

print('Model train rows:', len(model_train_df))
print('Internal eval rows:', len(internal_eval_df))
print('Frozen final val rows:', len(final_val_df))

In [ ]:
MODEL_NAME = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
MAX_LENGTH = 256
EPOCHS = 4
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
USE_FP16 = torch.cuda.is_available()
OUTPUT_DIR = Path('runs/biomedbert_mtsamples_colab')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('fp16:', USE_FP16)
print('output:', OUTPUT_DIR)

In [ ]:
class ClinicalTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt',
        )
        item = {key: value.squeeze(0) for key, value in encoded.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, labels=list(range(5)), average='macro', zero_division=0),
        'weighted_f1': f1_score(labels, preds, labels=list(range(5)), average='weighted', zero_division=0),
    }

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5,
    id2label=ID_TO_CLASS,
    label2id=CLASS_TO_LABEL,
)

train_dataset = ClinicalTextDataset(
    model_train_df['encoder_text'],
    model_train_df['hackathon_label'],
    tokenizer,
    MAX_LENGTH,
)
internal_eval_dataset = ClinicalTextDataset(
    internal_eval_df['encoder_text'],
    internal_eval_df['hackathon_label'],
    tokenizer,
    MAX_LENGTH,
)
final_val_dataset = ClinicalTextDataset(
    final_val_df['encoder_text'],
    final_val_df['hackathon_label'],
    tokenizer,
    MAX_LENGTH,
)

In [ ]:
training_kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=25,
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    report_to='none',
    seed=SEED,
    fp16=USE_FP16,
)

try:
    training_args = TrainingArguments(evaluation_strategy='epoch', **training_kwargs)
except TypeError:
    training_args = TrainingArguments(eval_strategy='epoch', **training_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=internal_eval_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

## Final Frozen Validation Prediction

This is the first time the notebook uses `val.csv` for model outputs. These are the numbers to report.

In [ ]:
internal_metrics = trainer.evaluate(internal_eval_dataset)
final_pred = trainer.predict(final_val_dataset)
final_pred_labels = np.argmax(final_pred.predictions, axis=1)

final_metrics = {
    'accuracy': accuracy_score(final_val_df['hackathon_label'], final_pred_labels),
    'macro_f1': f1_score(final_val_df['hackathon_label'], final_pred_labels, labels=list(range(5)), average='macro', zero_division=0),
    'weighted_f1': f1_score(final_val_df['hackathon_label'], final_pred_labels, labels=list(range(5)), average='weighted', zero_division=0),
}

print('Internal eval metrics used during training:')
print(internal_metrics)
print('\nFrozen val metrics for reporting:')
print(final_metrics)

print('\nPer-class report:')
print(classification_report(
    final_val_df['hackathon_label'],
    final_pred_labels,
    labels=list(range(5)),
    target_names=DISPLAY_CLASS_ORDER,
    zero_division=0,
))

print('Confusion matrix rows=true, cols=pred:')
print(pd.DataFrame(
    confusion_matrix(final_val_df['hackathon_label'], final_pred_labels, labels=list(range(5))),
    index=DISPLAY_CLASS_ORDER,
    columns=DISPLAY_CLASS_ORDER,
))

In [ ]:
predictions_df = pd.DataFrame({
    'id': final_val_df['id'].astype(str).to_numpy(),
    'prediction': [ID_TO_DISPLAY_CLASS[int(label)] for label in final_pred_labels],
})

pred_path = OUTPUT_DIR / 'validation_predictions.csv'
metrics_path = OUTPUT_DIR / 'metrics.json'
logits_path = OUTPUT_DIR / 'validation_logits.npy'
labels_path = OUTPUT_DIR / 'validation_labels.npy'

predictions_df.to_csv(pred_path, index=False)
np.save(logits_path, final_pred.predictions)
np.save(labels_path, final_val_df['hackathon_label'].to_numpy())

report_dict = classification_report(
    final_val_df['hackathon_label'],
    final_pred_labels,
    labels=list(range(5)),
    target_names=DISPLAY_CLASS_ORDER,
    zero_division=0,
    output_dict=True,
)

with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump({
        'internal_eval': internal_metrics,
        'frozen_val': final_metrics,
        'frozen_val_classification_report': report_dict,
        'protocol': {
            'train_file': str(TRAIN_FILE),
            'val_file': str(VAL_FILE),
            'val_used_for_training_or_early_stopping': False,
        },
    }, f, indent=2)

trainer.save_model(OUTPUT_DIR / 'best_model')
tokenizer.save_pretrained(OUTPUT_DIR / 'best_model')

print('Wrote EVAL-compatible predictions:', pred_path)
print('Wrote metrics:', metrics_path)
display(predictions_df.head())

In [ ]:
# Optional: download the key artifacts from Colab.
try:
    from google.colab import files
    files.download(str(pred_path))
    files.download(str(metrics_path))
except Exception as exc:
    print('Download helper skipped:', exc)